# **NLP Homework 2 — Text Representation**
**Yachay Tech University**

**Student:** Alessa Melo  
**Date:** May 23, 2026

---

## **Objective**

The goal of this assignment is to understand fundamental concepts in text mining and text representation, with a particular focus on **Term Frequency–Inverse Document Frequency (TF-IDF)**. Students will explore how raw text is transformed into numerical representations suitable for machine learning.

# **Part 2: Bag-of-Words Representation**

Consider the following corpus of 3 documents:

- **D1** = "I love natural language processing"
- **D2** = "Natural language processing is fun"
- **D3** = "I love machine learning"

## **2.1 Tokenization and Lowercase Conversion**

The code below applies **lowercase normalization** and **tokenization**.

- **Lowercase normalization** ensures consistency among words (e.g., `Natural` and `natural` are treated as the same token).
- **Tokenization** divides each document into individual lexical units (tokens) by splitting on whitespace.

The output shows the processed token list for each document.

In [ ]:
corpus = [
    "I love natural language processing",
    "Natural language processing is fun",
    "I love machine learning"
]

# Tokenization + lowercase conversion
doc_tokens = []
for doc in corpus:
    tokens = doc.lower().split()
    doc_tokens.append(tokens)

#Print 
for i, tokens in enumerate(doc_tokens, start=1):
    print(f"D{i}: {tokens}")

Tokenized Documents:
D1: ['i', 'love', 'natural', 'language', 'processing']
D2: ['natural', 'language', 'processing', 'is', 'fun']
D3: ['i', 'love', 'machine', 'learning']
[['i', 'love', 'natural', 'language', 'processing'], ['natural', 'language', 'processing', 'is', 'fun'], ['i', 'love', 'machine', 'learning']]


## **2.2 Building the Vocabulary**

The **vocabulary** is the set of all unique tokens found across the entire corpus. Each document is iterated and tokens not yet in the vocabulary are appended, preserving insertion order.

For this corpus the vocabulary contains **|V| = 9** unique terms:

> `[i, love, natural, language, processing, is, fun, machine, learning]`

In [ ]:
# Build vocabulary
vocabulary = []
for document in doc_tokens:
    for word in document:
        if word not in vocabulary:
            vocabulary.append(word)

print("Vocabulary:",vocabulary)

Vocabulary:
['i', 'love', 'natural', 'language', 'processing', 'is', 'fun', 'machine', 'learning']


## **2.3 Bag-of-Words (BoW) Term-Document Matrix**

The **Bag-of-Words** model represents each document as a vector of raw term counts over the vocabulary. For every document, `Counter` tallies how many times each vocabulary word appears, and the result is stored as a row in the matrix. The matrix is then transposed so that rows are terms and columns are documents.

Each cell contains the **raw count** of the term (row) in the document (column):

| Term | D1 | D2 | D3 |
|------|----|----|----|
| i | 1 | 0 | 1 |
| love | 1 | 0 | 1 |
| natural | 1 | 1 | 0 |
| language | 1 | 1 | 0 |
| processing | 1 | 1 | 0 |
| is | 0 | 1 | 0 |
| fun | 0 | 1 | 0 |
| machine | 0 | 0 | 1 |
| learning | 0 | 0 | 1 |

> **Note:** BoW ignores word order and sentence structure — it only captures how often each term appears. This is both its simplicity and its main limitation.

In [12]:
from collections import Counter
import pandas as pd

# Construct Bag-of-Words matrix
bow_matrix = []
index =["D1", "D2", "D3"]

for document in doc_tokens:
    word_counts = Counter(document)

    row = []
    for word in vocabulary:
        row.append(word_counts[word])

    bow_matrix.append(row)

# Convert to term-document matrix
bow_df = pd.DataFrame(bow_matrix, columns=vocabulary, index=index)

print("Bag-of-Words Representation:")
print(bow_df.T)

Bag-of-Words Representation:
            D1  D2  D3
i            1   0   1
love         1   0   1
natural      1   1   0
language     1   1   0
processing   1   1   0
is           0   1   0
fun          0   1   0
machine      0   0   1
learning     0   0   1



# **Part 3: Term Frequency (TF)**

**Term Frequency** TF(t, d) measures how frequently a term *t* appears in a document *d*. The main idea is that words appearing more often in a document are usually more important for describing its content.

## **3.1 Raw Term Frequency**

The raw TF is simply the count of how many times a term appears in the document — identical to the BoW matrix computed above. Document lengths for this corpus are:

- |D1| = 5, |D2| = 5, |D3| = 4

## **3.2 Compute the Raw Term Frequency for Each Term in Each Document**

In [14]:
from collections import Counter
import pandas as pd

# Compute raw term frequency
tf_matrix = []

for document in doc_tokens:
    word_counts = Counter(document)

    row = []
    for word in vocabulary:
        row.append(word_counts[word])

    tf_matrix.append(row)

# Convert to DataFrame
tf_df = pd.DataFrame(tf_matrix, columns=vocabulary, index=index)

print("Raw Term Frequency Matrix:")
print(tf_df.T)

Raw Term Frequency Matrix:
            D1  D2  D3
i            1   0   1
love         1   0   1
natural      1   1   0
language     1   1   0
processing   1   1   0
is           0   1   0
fun          0   1   0
machine      0   0   1
learning     0   0   1


## **3.3 Normalized Term Frequency**

Instead of raw counts, TF is typically **normalized** by the total number of terms in the document:

$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t'} f_{t',d}}$$

where $f_{t,d}$ is the count of term $t$ in document $d$, and the denominator is the total number of tokens in that document.

This converts raw counts into **proportions** (values between 0 and 1), making documents of different lengths comparable. For example:
- Any term appearing once in D1 (5 tokens) → TF = 1/5 = **0.2000**
- Any term appearing once in D3 (4 tokens) → TF = 1/4 = **0.2500**

> **Why is normalization useful?** Without it, longer documents would systematically have higher raw counts than shorter ones for the same proportion of term usage, making cross-document comparisons unfair.

In [19]:
for i, row in enumerate(tf_matrix):
    total_terms = sum(row)
    tf_matrix[i] = [count / total_terms for count in row]

print("Normalized Term Frequency Matrix:")
tf_df = pd.DataFrame(tf_matrix, columns=vocabulary, index=index)
print(tf_df.T)


Normalized Term Frequency Matrix:
             D1   D2    D3
i           0.2  0.0  0.25
love        0.2  0.0  0.25
natural     0.2  0.2  0.00
language    0.2  0.2  0.00
processing  0.2  0.2  0.00
is          0.0  0.2  0.00
fun         0.0  0.2  0.00
machine     0.0  0.0  0.25
learning    0.0  0.0  0.25


# **Part 4: Inverse Document Frequency (IDF)**

**Inverse Document Frequency** IDF(t) measures how **informative or unique** a term *t* is across the entire corpus. The intuition is:
- Words appearing in **many documents** → less useful for distinguishing documents → **low IDF**
- Words appearing in **few documents** → more specific and discriminative → **high IDF**

The IDF formula is:

$$\text{IDF}(t) = \log\left(\frac{N}{df_t}\right)$$

where:
- $N$ = total number of documents in the corpus
- $df_t$ = number of documents containing term $t$ at least once

The **logarithm** reduces the scale of values and prevents extremely large weights for very rare terms.

> **Special case:** If a term appears in every document ($df_t = N$), then IDF = log(N/N) = log(1) = **0**, contributing nothing to the final score. This naturally down-weights stopwords.

## **4.1 Computing IDF for Each Term**

With N = 3 documents, the expected values are:

| Term | df | IDF = ln(3/df) |
|------|----|-----------------|
| fun, is, learning, machine | 1 | ln(3) ≈ **1.0986** |
| i, love, natural, language, processing | 2 | ln(3/2) ≈ **0.4055** |

Terms appearing in only **one** document (fun, is, learning, machine) receive the **highest IDF** — they are the most distinctive.

In [ ]:
import math
import pandas as pd

# Total number of documents
N = len(doc_tokens)

# Compute document frequency (df)
df = {}

for word in vocabulary:
    count = 0

    for document in doc_tokens:
        if word in document:
            count += 1

    df[word] = count

# Compute IDF
idf = {}

for word in vocabulary:
    idf[word] = math.log(N / df[word])

# Convert to DataFrame
idf_df = pd.DataFrame.from_dict(idf, orient='index', columns=['IDF'])

print("Inverse Document Frequency (IDF):")
print(idf_df)

Inverse Document Frequency (IDF):
                 IDF
i           0.405465
love        0.405465
natural     0.405465
language    0.405465
processing  0.405465
is          1.098612
fun         1.098612
machine     1.098612
learning    1.098612


# **Part 5: TF-IDF Representation**

**TF-IDF** (Term Frequency – Inverse Document Frequency) combines both measures to score how important a word is to a specific document relative to the whole corpus:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

- A **high TF-IDF** score means the term is frequent in this document *and* rare across the corpus → very characteristic of this document.
- A **low TF-IDF** score means the term is common across many documents → less discriminative.

TF-IDF gives a high score to words that are **frequent in a specific document but rare in the overall collection**, making them useful for identifying the document's main topic.

##  **Compute the TF-IDF Score for Each Term in Each Document**

In [ ]:
import pandas as pd

tfidf_matrix = []
for document in doc_tokens:
    word_counts = Counter(document)
    total_terms = len(document)
    row = []
    for word in vocabulary:
        tf = word_counts[word] / total_terms
        # TF-IDF
        tfidf = tf * idf[word]
        row.append(tfidf)
    tfidf_matrix.append(row)

tfidf_df = pd.DataFrame(
    tfidf_matrix,
    columns=vocabulary,
    index=index
)

print("TF-IDF Matrix:")
print(tfidf_df.T)

TF-IDF Matrix:
                  D1        D2        D3
i           0.081093  0.000000  0.101366
love        0.081093  0.000000  0.101366
natural     0.081093  0.081093  0.000000
language    0.081093  0.081093  0.000000
processing  0.081093  0.081093  0.000000
is          0.000000  0.219722  0.000000
fun         0.000000  0.219722  0.000000
machine     0.000000  0.000000  0.274653
learning    0.000000  0.000000  0.274653


### **Interpreting the TF-IDF Matrix**

The most important term(s) per document according to TF-IDF:

| Document | Top Term(s) | TF-IDF Score | Reason |
|----------|-------------|--------------|--------|
| **D1** | i, language, love, natural, processing (tie) | 0.0811 | All content terms appear only once; none is uniquely exclusive to D1 |
| **D2** | **fun**, **is** | 0.2197 | Appear only in D2, giving them the highest IDF in the corpus |
| **D3** | **learning**, **machine** | 0.2747 | Appear only in D3, and D3 is shorter (TF = 0.25 vs 0.20) |

> **Why TF-IDF works better than raw frequency:** Raw frequency favors any word that appears often, even uninformative stopwords. TF-IDF introduces IDF to reduce the importance of words common across many documents while elevating terms that are rare and distinctive — highlighting the actual topic of each document.

# **Part 6: Practical Implementation with scikit-learn**

##  **6.1 Use `TfidfVectorizer` to Compute TF-IDF for the Corpus**

scikit-learn's `TfidfVectorizer` automates the full pipeline. However, it uses a **slightly modified IDF formula** with smoothing to prevent division by zero:

$$\text{IDF}(t) = \log\left(\frac{1 + N}{1 + df_t}\right) + 1$$

It also applies **L2 normalization** on each document vector after computing TF-IDF scores:

$$v_{\text{norm}} = \frac{v}{\|v\|_2}$$

This scales each vector to unit Euclidean norm, which explains why the library values differ from the manual calculation. Additionally, by default `TfidfVectorizer` **excludes single-character tokens** (like `"i"`), which is why it does not appear in the output below.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer()
# Compute TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(corpus)

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=index
)

print("TF-IDF Representation using TfidfVectorizer:")
print(tfidf_df.T)

TF-IDF Representation using TfidfVectorizer:
             D1        D2        D3
fun         0.0  0.517420  0.000000
is          0.0  0.517420  0.000000
language    0.5  0.393511  0.000000
learning    0.0  0.000000  0.622766
love        0.5  0.000000  0.473630
machine     0.0  0.000000  0.622766
natural     0.5  0.393511  0.000000
processing  0.5  0.393511  0.000000


# **Experiments: Preprocessing Variations**

The following three experiments explore how different preprocessing choices affect the TF-IDF representation.

## **Experiment 1: Remove Stopwords**

**Stopwords** are highly frequent words (e.g., *the*, *is*, *a*, *and*, *i*) that carry little semantic content. Removing them:
- Reduces noise and vocabulary size
- Allows models to focus on more informative, content-bearing words
- Increases the TF-IDF weights of the remaining meaningful terms

Here, `stop_words="english"` removes the built-in English stopword list. The `token_pattern=r"(?u)\b\w+\b"` setting is used so that single-character tokens like `"i"` are also considered (and then filtered as a stopword).

In [25]:
# Same corpus
corpus = [
    "I love natural language processing",
    "Natural language processing is fun",
    "I love machine learning"
]

index = ["D1", "D2", "D3"]

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vec_sw = TfidfVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b\w+\b"
)

X_sw = vec_sw.fit_transform(corpus)

df_sw = pd.DataFrame(
    X_sw.toarray(),
    index=index,
    columns=vec_sw.get_feature_names_out(),
)

print("=== With stopword removal ===")
print(df_sw.round(4).T)

=== With stopword removal ===
             D1      D2      D3
fun         0.0  0.6047  0.0000
language    0.5  0.4599  0.0000
learning    0.0  0.0000  0.6228
love        0.5  0.0000  0.4736
machine     0.0  0.0000  0.6228
natural     0.5  0.4599  0.0000
processing  0.5  0.4599  0.0000


## **Experiment 2: Unigrams + Bigrams (N-Grams)**

**N-grams** extend the representation from isolated words to short word sequences. Setting `ngram_range=(1, 2)` includes both **unigrams** (single words) and **bigrams** (pairs of consecutive words).

Benefits of using bigrams:
- Captures contextual meaning and meaningful phrases (e.g., *machine learning*, *natural language*)
- Preserves some word-order information that unigrams ignore

Trade-offs:
- The vocabulary grows significantly (from 9 to 17 features here)
- The resulting matrix is **sparser** since most bigrams appear in only one document
- Higher memory and computational cost for large corpora

In [ ]:
# --- Experiment 2: Unigrams + bigrams ---

vec_ng = TfidfVectorizer(
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+\b"
)

X_ng = vec_ng.fit_transform(corpus)

df_ng = pd.DataFrame(
    X_ng.toarray(),
    index=index,
    columns=vec_ng.get_feature_names_out(),
)

print("\n=== With bigrams ===")
print(df_ng.round(4).T)


=== With bigrams ===
                         D1      D2      D3
fun                  0.0000  0.3809  0.0000
i                    0.3206  0.0000  0.3176
i love               0.3206  0.0000  0.3176
is                   0.0000  0.3809  0.0000
is fun               0.0000  0.3809  0.0000
language             0.3206  0.2897  0.0000
language processing  0.3206  0.2897  0.0000
learning             0.0000  0.0000  0.4176
love                 0.3206  0.0000  0.3176
love machine         0.0000  0.0000  0.4176
love natural         0.4216  0.0000  0.0000
machine              0.0000  0.0000  0.4176
machine learning     0.0000  0.0000  0.4176
natural              0.3206  0.2897  0.0000
natural language     0.3206  0.2897  0.0000
processing           0.3206  0.2897  0.0000
processing is        0.0000  0.3809  0.0000


## **Experiment 3: Stopword Removal + Bigrams (Combined)**

Combining both preprocessing strategies produces the **most semantically meaningful representation** in this experiment:

- **Stopword removal** eliminates low-information words (*i*, *is*), so the vocabulary focuses on content words.
- **Bigrams** preserve contextual phrases (*love natural*, *natural language*, *machine learning*), adding more descriptive power than unigrams alone.

The result is a compact vocabulary of high-quality features that better characterizes each document's topic.

In [ ]:
# --- Experiment 3: Stopword removal + bigrams combined ---
vec_both = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+\b"
)

X_both = vec_both.fit_transform(corpus)

df_both = pd.DataFrame(
    X_both.toarray(),
    index=index,
    columns=vec_both.get_feature_names_out(),
)

print("\n=== Stopwords removed + bigrams ===")
print(df_both.round(4).T)


=== Stopwords removed + bigrams ===
                         D1      D2      D3
fun                  0.0000  0.4521  0.0000
language             0.3597  0.3439  0.0000
language processing  0.3597  0.3439  0.0000
learning             0.0000  0.0000  0.4674
love                 0.3597  0.0000  0.3554
love machine         0.0000  0.0000  0.4674
love natural         0.4730  0.0000  0.0000
machine              0.0000  0.0000  0.4674
machine learning     0.0000  0.0000  0.4674
natural              0.3597  0.3439  0.0000
natural language     0.3597  0.3439  0.0000
processing           0.3597  0.3439  0.0000
processing fun       0.0000  0.4521  0.0000


## **Summary of Experiments**

| Configuration | Vocab Size | Key Effect |
|--------------|------------|------------|
| Baseline (unigrams) | 8 terms | Includes stopwords; `i` excluded by default pattern |
| + Stopword removal | 7 terms | Removes `i`, `is`; remaining terms get higher weights |
| + Bigrams only | 17 terms | Adds phrase-level features; larger, sparser matrix |
| + Both combined | 13 terms | Best balance: content-focused + contextual phrases |

> **Conclusion:** Combining stopword removal with bigrams produces the most semantically meaningful TF-IDF representation. The vocabulary focuses on relevant concepts while preserving contextual phrases, improving the descriptive power of the feature vectors.